# Serving your model

Before you start:

1. `uv add fastapi uvicorn`
2. In an iTerm2 tab, from `ML_Lab`, start the model server and leave it running:

```bash
uv run mlx_lm.server --model lessons/07_mlx_lora/models/darvis-router-1.5b-4bit --port 8080
```

In [ ]:
import asyncio
import json
import os
import re
import statistics
import time

import httpx
import numpy as np
from openai import AsyncOpenAI, OpenAI

MODEL_URL = "http://localhost:8080/v1"
MODEL = "default_model"     # mlx_lm.server maps this name to the model it was started with
SYSTEM = "You are the Darvis query router. Convert the student's question into a JSON query."

client = OpenAI(base_url=MODEL_URL, api_key="local")
aclient = AsyncOpenAI(base_url=MODEL_URL, api_key="local")

def ask(question, temperature=0.0):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        temperature=temperature,
        max_tokens=100,
    )
    return r

r = ask("who's easier for data structures, alvarez or chen? fall '26")
print(r.choices[0].message.content)
print(r.usage)

That's the same `openai` client and the same code shape as module 01. Only the URL changed. Your fine-tuned model is now a drop-in replacement for any OpenAI-compatible model in any app.

## 1. Latency

Users feel latency, and averages hide the slow requests. Measure percentiles: p50 is the typical request, and p95 is the experience of your unluckiest 1 in 20 users.

In [ ]:
QUESTIONS = [
    "who's easier for data structures, alvarez or chen? fall '26",
    "prereqs for cs3214?",
    "what is lin alg about",
    "best professor for comp org 2 next spring",
    "is okafor teaching systems this fall",
    "avg gpa MATH 2204",
    "compare rossi and haddad in algorithms",
    "where can i print on campus",
    "how much work is the ml class",
    "can i take cs 3114 without CS 2114",
    "who teaches dsa best",
    "is CS 4104 with Dr. Lindqvist a gpa booster",
    "when is mobile dev usually offered",
    "Nakamura or Duarte for stats for engineers?",
    "what topics does comp sys cover",
    "how do i reset my vt password",
]

latencies, out_tokens = [], []
for q in QUESTIONS:
    start = time.perf_counter()
    r = ask(q)
    latencies.append(time.perf_counter() - start)
    out_tokens.append(r.usage.completion_tokens if r.usage else len(r.choices[0].message.content) / 4)

lat_ms = np.array(latencies) * 1000
print(f"requests: {len(lat_ms)}")
print(f"p50 {np.percentile(lat_ms, 50):.0f} ms   p95 {np.percentile(lat_ms, 95):.0f} ms   max {lat_ms.max():.0f} ms")
print(f"average output: {np.mean(out_tokens):.0f} tokens   sequential throughput: {len(lat_ms) / sum(latencies):.2f} requests/s")

## 2. Concurrency

Real traffic arrives in parallel. Send the same questions with 1, 4, and 8 requests in flight at once and watch what happens to throughput and to per-request latency.

In [ ]:
async def timed_request(question, semaphore):
    async with semaphore:
        start = time.perf_counter()
        await aclient.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
            temperature=0.0,
            max_tokens=100,
        )
        return time.perf_counter() - start

async def load_test(concurrency, n_requests=32):
    semaphore = asyncio.Semaphore(concurrency)
    questions = [QUESTIONS[i % len(QUESTIONS)] for i in range(n_requests)]
    start = time.perf_counter()
    times = await asyncio.gather(*(timed_request(q, semaphore) for q in questions))
    wall = time.perf_counter() - start
    ms = np.array(times) * 1000
    print(f"concurrency {concurrency:2}: {n_requests / wall:5.2f} req/s   p50 {np.percentile(ms, 50):6.0f} ms   p95 {np.percentile(ms, 95):6.0f} ms")

for c in [1, 4, 8]:
    await load_test(c)

How to read this. If throughput rises with concurrency, the server is batching: it runs several requests through the model together, which uses the hardware better. If throughput stays flat while latency climbs, requests are just queueing. `mlx_lm.server` batches to a degree on Apple Silicon, while production servers like vLLM are built around continuous batching and scale much further on a GPU.

The number to carry forward is your peak requests per second. It's the capacity of one Mac (or one GPU) serving this model.

## 3. The backend service

A raw model endpoint isn't safe to expose to users. It can return malformed JSON, invent intents, or hang. `app.py` in this folder wraps it with:

- Input validation (Pydantic rejects empty or huge questions before they reach the model)
- Output validation against the exact schema, including the allowed intents and terms
- One retry on bad output, then a safe fallback instead of an error
- Timing on every request

Start it now in a second iTerm2 tab:

```bash
cd lessons/08_serving
uv run uvicorn app:app --port 8000 --reload
```

Then open http://localhost:8000/docs in your browser and try the endpoint there.

In [ ]:
API = "http://localhost:8000"

print(httpx.get(f"{API}/health", timeout=10).json())
print()
for q in QUESTIONS[:6]:
    r = httpx.post(f"{API}/route", json={"question": q}, timeout=30).json()
    print(f"{q!r}\n  -> {r['route']}  attempts={r['attempts']} fallback={r['fallback']} {r['latency_ms']:.0f} ms")

In [ ]:
# Bad input never reaches the model. FastAPI answers with a 422 and an explanation.
r = httpx.post(f"{API}/route", json={"question": ""}, timeout=30)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:400])

## 4. What does it cost?

Three options for running the router:

1. A hosted API (OpenAI, Anthropic, Together, and others): you pay per token, nothing when idle, and it scales automatically.
2. A rented GPU running vLLM: a fixed hourly price whether you use it or not, with capacity set by throughput.
3. Your Mac: effectively free, but it isn't production. It sleeps, it's on home internet, and it's your laptop.

The prices below are placeholders. Look up current prices for a hosted model you'd actually consider and for a GPU you'd rent, and replace them. The structure of the answer matters more than these numbers.

In [ ]:
# Measured from your model (section 1). Input tokens include the chat template around the short system prompt.
avg_in_tokens = 60
avg_out_tokens = float(np.mean(out_tokens))

# Placeholders: replace with real prices
HOSTED_IN_PRICE = 0.50     # $ per 1M input tokens
HOSTED_OUT_PRICE = 1.50    # $ per 1M output tokens
GPU_HOURLY = 0.60          # $ per hour for a rented GPU
GPU_REQ_PER_SEC = 20       # replace with a measured vLLM number; it's typically far above your Mac's

def hosted_monthly(requests):
    return requests * (avg_in_tokens * HOSTED_IN_PRICE + avg_out_tokens * HOSTED_OUT_PRICE) / 1e6

gpu_monthly = GPU_HOURLY * 730   # always-on
gpu_capacity = GPU_REQ_PER_SEC * 3600 * 730

for requests in [10_000, 100_000, 1_000_000, 10_000_000]:
    print(f"{requests:>12,} req/month   hosted ${hosted_monthly(requests):>10,.2f}   one GPU ${gpu_monthly:>8,.2f}"
          + ("" if requests <= gpu_capacity else "  (over one GPU's capacity)"))

per_request = hosted_monthly(1)
print(f"\nbreak-even: about {gpu_monthly / per_request:,.0f} requests/month")

Notice what this shows. Because your fine-tuned model needs only a one-line prompt and a 1.5B model is cheap per token, hosted per-request cost is tiny, so a dedicated GPU only wins at high volume. That's typical: self-hosting usually wins on control, privacy, latency, and owning the model, and only wins on price at scale. The fine-tuning itself is what made the per-request cost small, whichever way you host it.

For Darvis today, a reasonable plan: the router runs on whatever is cheapest to operate at current traffic (a hosted endpoint serving your fine-tuned model, or a small GPU instance), behind the same FastAPI contract, so you can switch later by changing `MODEL_BASE_URL`.

---
# Exercises

**Exercise 1: Caching.** Students ask the same questions constantly. Add an in-memory cache to `app.py`: normalize the question (lowercase, collapse whitespace), and if it's been seen, return the stored route without calling the model. Add a `cached: bool` field to the response. Measure latency for a repeated question before and after.

**Exercise 2: Logging for future training data.** Make `app.py` append one JSON line per request to `logs/requests.jsonl` with a timestamp, the question, the route, attempts, fallback, and latency. This file is the seed of your next, better training set (module 07, exercise 2). Before doing this with real users, decide what you'll tell them and how long you'll keep it. Questions can contain personal information. Add `logs/` to `.gitignore`.

**Exercise 3: Load test the service.** Using `httpx.AsyncClient`, send 64 requests to `/route` with 8 in flight at a time. Report requests per second, p50 and p95 latency, and the fallback rate. Compare with the raw model numbers from section 2. How much overhead does the FastAPI layer add?

**Exercise 4: Fine-tuned vs. hosted.** If you have an API key for a hosted model, score it on module 07's test set (`../07_mlx_lora/data/test.jsonl`) using the detailed system prompt from module 07, and compare exact-match accuracy, p50 latency, and estimated cost per 1,000 requests with your fine-tuned model.

**Exercise 5: Deployment plan.** Write half a page, for yourself, on how the router would ship inside Darvis: where the model runs, how the FastAPI router is called from the existing backend, what happens when the model server is down, how you'll monitor accuracy after launch, and the monthly cost at your current traffic. This is the kind of document you'll write for every infrastructure decision as a founder.

---
# Solutions

Exercises 1 and 2 are edits to `app.py`. Here's the changed part of the `/route` endpoint, with both a cache and logging. Replace the `route` function in `app.py` with this, add `cached: bool = False` to `RouteResponse`, and add `import datetime` at the top:

```python
CACHE: dict[str, Route] = {}
os.makedirs("logs", exist_ok=True)

def normalize(q: str) -> str:
    return re.sub(r"\s+", " ", q.strip().lower())

def log_request(question, result, attempts, fallback, latency_ms, cached):
    with open("logs/requests.jsonl", "a") as f:
        f.write(json.dumps({
            "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "question": question, "route": result.model_dump(), "attempts": attempts,
            "fallback": fallback, "latency_ms": round(latency_ms, 1), "cached": cached,
        }) + "\n")

@app.post("/route", response_model=RouteResponse)
async def route(body: Question):
    start = time.perf_counter()
    key = normalize(body.question)
    if key in CACHE:
        latency = (time.perf_counter() - start) * 1000
        log_request(body.question, CACHE[key], 0, False, latency, True)
        return RouteResponse(route=CACHE[key], attempts=0, fallback=False, latency_ms=latency, cached=True)

    for attempt in range(1, MAX_ATTEMPTS + 1):
        response = await client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": body.question}],
            temperature=0.0, max_tokens=100,
        )
        parsed = extract_json(response.choices[0].message.content or "")
        if parsed is not None:
            try:
                result = Route.model_validate(parsed)
                CACHE[key] = result
                latency = (time.perf_counter() - start) * 1000
                log_request(body.question, result, attempt, False, latency, False)
                return RouteResponse(route=result, attempts=attempt, fallback=False, latency_ms=latency)
            except ValidationError:
                pass
    latency = (time.perf_counter() - start) * 1000
    log_request(body.question, FALLBACK, MAX_ATTEMPTS, True, latency, False)
    return RouteResponse(route=FALLBACK, attempts=MAX_ATTEMPTS, fallback=True, latency_ms=latency)
```

A production cache would live in Redis with an expiry so it's shared across server processes and survives restarts. The test below works once your edited `app.py` has reloaded.

In [ ]:
# Exercise 1 check
q = {"question": "prereqs for cs3214?"}
for i in range(3):
    r = httpx.post(f"{API}/route", json=q, timeout=30).json()
    print(f"call {i + 1}: {r['latency_ms']:7.1f} ms  cached={r.get('cached')}")

In [ ]:
# Exercise 3
async def load_test_api(n_requests=64, concurrency=8):
    semaphore = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(timeout=60) as http:
        async def one(i):
            async with semaphore:
                start = time.perf_counter()
                r = await http.post(f"{API}/route", json={"question": QUESTIONS[i % len(QUESTIONS)] + f" ({i})"})
                return time.perf_counter() - start, r.json()["fallback"]
        start = time.perf_counter()
        results = await asyncio.gather(*(one(i) for i in range(n_requests)))
        wall = time.perf_counter() - start
    ms = np.array([t for t, _ in results]) * 1000
    fallbacks = sum(f for _, f in results)
    print(f"{n_requests / wall:.2f} req/s   p50 {np.percentile(ms, 50):.0f} ms   p95 {np.percentile(ms, 95):.0f} ms   fallback rate {fallbacks / n_requests:.0%}")

# The " (i)" suffix makes every question unique so a cache from exercise 1 doesn't inflate the numbers.
await load_test_api()

In [ ]:
# Exercise 4
HOSTED_URL = os.getenv("HOSTED_BASE_URL")        # for example https://api.openai.com/v1
HOSTED_KEY = os.getenv("HOSTED_API_KEY")
HOSTED_MODEL = os.getenv("HOSTED_MODEL")         # a small, cheap model is the fair comparison

def extract_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def exact_match(pred, target):
    if not isinstance(pred, dict):
        return False
    norm = lambda xs: sorted(str(x).strip().lower() for x in (xs or []))
    return (pred.get("intent") == target["intent"] and norm(pred.get("courses")) == norm(target["courses"])
            and norm(pred.get("professors")) == norm(target["professors"]) and pred.get("term") == target["term"])

with open("../07_mlx_lora/data/test.jsonl") as f:
    test_rows = [json.loads(line)["messages"] for line in f]

def evaluate(c, model, system, rows):
    hits, lat = 0, []
    for msgs in rows:
        start = time.perf_counter()
        r = c.chat.completions.create(model=model, temperature=0, max_tokens=100,
                                      messages=[{"role": "system", "content": system}, msgs[1]])
        lat.append(time.perf_counter() - start)
        hits += exact_match(extract_json(r.choices[0].message.content), json.loads(msgs[2]["content"]))
    return hits / len(rows), np.percentile(np.array(lat) * 1000, 50)

acc, p50 = evaluate(client, MODEL, SYSTEM, test_rows[:50])
print(f"fine-tuned (local): exact {acc:.0%}   p50 {p50:.0f} ms")

if HOSTED_URL and HOSTED_KEY and HOSTED_MODEL:
    DETAILED = "PASTE SYSTEM_DETAILED FROM MODULE 07 HERE"
    hosted = OpenAI(base_url=HOSTED_URL, api_key=HOSTED_KEY)
    acc_h, p50_h = evaluate(hosted, HOSTED_MODEL, DETAILED, test_rows[:50])
    print(f"hosted:             exact {acc_h:.0%}   p50 {p50_h:.0f} ms")
else:
    print("Set HOSTED_BASE_URL, HOSTED_API_KEY and HOSTED_MODEL before starting Jupyter to run the hosted comparison.")

Exercise 5 has no code. Write it in `experiments/` or Notion, and revisit it once Darvis has real traffic numbers.